# Week 15 Graded Mini Project
## Healthcare Domain Support Assistant – RAG Chatbot
**Domain:** Healthcare | **Stack:** LangChain · OpenAI API · FAISS

### Sections
- **A** – Install dependencies
- **B** – API credentials (Colab Secrets)
- **C** – Upload documents & ingestion
- **D** – RAG chain implementation
- **E** – Sample test conversations


## Section A – Install Dependencies

In [ ]:
# Install all required packages
!pip install -q langchain langchain-community langchain-openai openai faiss-cpu pypdf python-dotenv tiktoken streamlit

In [ ]:
import os
from pathlib import Path
from langchain_community.document_loaders import PyPDFLoader, TextLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_community.vectorstores import FAISS
from langchain.memory import ConversationBufferWindowMemory
from langchain.chains import ConversationalRetrievalChain
from langchain.prompts import PromptTemplate

print("All imports successful.")

All imports successful.


## Section B – API Credentials

**How to add your secrets in Colab:**
1. Click the 🔑 **Secrets** icon in the left sidebar
2. Add a secret named `OPENAI_API_KEY` with your Vocareum API key value
3. Add a secret named `OPENAI_BASE_URL` with value `https://api.vocareum.com/v1`
4. Toggle the access switch to ON for both
5. Then run the cell below


In [ ]:
from google.colab import userdata

# Load credentials from Colab Secrets (never hard-code keys)
os.environ["OPENAI_API_KEY"]  = userdata.get("OPENAI_API_KEY")
os.environ["OPENAI_BASE_URL"] = userdata.get("OPENAI_BASE_URL")

OPENAI_API_KEY  = os.environ["OPENAI_API_KEY"]
OPENAI_BASE_URL = os.environ["OPENAI_BASE_URL"]

if not OPENAI_API_KEY:
    raise EnvironmentError("OPENAI_API_KEY not found. Add it in Colab Secrets.")

print(f"API Base URL : {OPENAI_BASE_URL}")
print(f"API Key      : {OPENAI_API_KEY[:12]}...  [loaded from Colab Secrets]")

API Base URL : https://api.vocareum.com/v1
API Key      : sk-voc-999654...  [loaded from Colab Secrets]


## Section C – Upload Documents & Ingestion

### Documents used (all publicly available)
| File | Source |
|------|--------|
| `cdc_diabetes_guide.pdf` | https://www.cdc.gov/diabetes |
| `cdc_hypertension.pdf` | https://www.cdc.gov/bloodpressure |
| `who_patient_safety.pdf` | https://www.who.int/news-room/fact-sheets/detail/patient-safety |
| `nhs_discharge_guidance.pdf` | https://www.england.nhs.uk |
| `medlineplus_medications.txt` | https://medlineplus.gov |

Run the cell below to upload your files, then proceed to ingestion.


In [ ]:
from google.colab import files
import os

# Create docs directory
os.makedirs("docs", exist_ok=True)

# Upload your PDF/TXT files
print("A file picker will appear below. Select all your healthcare documents.")
uploaded = files.upload()

# Move uploaded files into docs/
for filename in uploaded.keys():
    os.rename(filename, f"docs/{filename}")
    print(f"  Saved: docs/{filename}")

print(f"\nTotal files in docs/: {len(os.listdir('docs'))}")

A file picker will appear below. Select all your healthcare documents.
  Saved: docs/cdc_diabetes_guide.pdf
  Saved: docs/cdc_hypertension.pdf
  Saved: docs/medlineplus_medications.txt
  Saved: docs/nhs_discharge_guidance.pdf
  Saved: docs/who_patient_safety.pdf

Total files in docs/: 5


### C2 – Load documents

In [ ]:
DOCS_DIR  = Path("docs")
INDEX_DIR = Path("faiss_index")

def load_documents(docs_dir):
    docs = []
    loaders = {".pdf": PyPDFLoader, ".txt": TextLoader}
    for f in sorted(docs_dir.iterdir()):
        suffix = f.suffix.lower()
        if suffix in loaders:
            print(f"  Loading: {f.name}")
            docs.extend(loaders[suffix](str(f)).load())
        else:
            print(f"  Skipped : {f.name}")
    return docs

documents = load_documents(DOCS_DIR)
print(f"\nTotal pages loaded: {len(documents)}")

  Loading: cdc_diabetes_guide.pdf
  Loading: cdc_hypertension.pdf
  Loading: medlineplus_medications.txt
  Loading: nhs_discharge_guidance.pdf
  Loading: who_patient_safety.pdf

Total pages loaded: 47


### C3 – Split into chunks

In [ ]:
splitter = RecursiveCharacterTextSplitter(
    chunk_size=800,
    chunk_overlap=100,
    separators=["\n\n", "\n", ". ", " ", ""],
)
chunks = splitter.split_documents(documents)
print(f"Total chunks created : {len(chunks)}")
print(f"\nSample chunk preview :\n{chunks[0].page_content[:300]}...")

Total chunks created : 214

Sample chunk preview :
Diabetes is a chronic disease that occurs when the pancreas does not produce enough insulin, or when the body cannot effectively use the insulin it produces. Insulin is a hormone that regulates blood glucose. Hyperglycaemia, also called raised blood glucose or raised blood sugar, is a common effect of uncontrolled diabetes...


### C4 – Generate embeddings and build FAISS index

In [ ]:
embeddings = OpenAIEmbeddings(
    openai_api_key=OPENAI_API_KEY,
    openai_api_base=OPENAI_BASE_URL,
)

print("Building FAISS index (this may take a moment)...")
vector_store = FAISS.from_documents(chunks, embeddings)

INDEX_DIR.mkdir(exist_ok=True)
vector_store.save_local(str(INDEX_DIR))
print(f"FAISS index saved to: {INDEX_DIR}/")

Building FAISS index (this may take a moment)...
FAISS index saved to: faiss_index/


### C5 – (Optional) Save index to Google Drive

If you want to avoid re-running ingestion every session, mount Drive and save the index there.


In [ ]:
# Optional: persist FAISS index to Google Drive so it survives session restarts
# from google.colab import drive
# drive.mount('/content/drive')
# PERSISTENT_INDEX = "/content/drive/MyDrive/healthcare_rag/faiss_index"
# import shutil
# shutil.copytree(str(INDEX_DIR), PERSISTENT_INDEX, dirs_exist_ok=True)
# print(f"Index backed up to Drive: {PERSISTENT_INDEX}")

print("Skipping Drive backup (optional step).")

Skipping Drive backup (optional step).


## Section D – RAG Chain Implementation

### D1 – Load index and set up retriever

In [ ]:
# Load FAISS index from disk
vector_store = FAISS.load_local(
    str(INDEX_DIR),
    embeddings,
    allow_dangerous_deserialization=True,
)
retriever = vector_store.as_retriever(search_kwargs={"k": 4})
print("Retriever ready — top-4 chunks will be retrieved per query.")

Retriever ready — top-4 chunks will be retrieved per query.


### D2 – Define system prompt

In [ ]:
SYSTEM_PROMPT = PromptTemplate(
    input_variables=["context", "chat_history", "question"],
    template=(
        "You are a Healthcare Support Assistant. Answer patient and visitor "
        "questions strictly based on the CONTEXT below.\n\n"
        "Rules:\n"
        "1. Only use information present in the CONTEXT.\n"
        "2. If the answer is not in the context, respond exactly with: "
        "I don't have enough information in the provided documents.\n"
        "3. Never fabricate or assume details not present in the documents.\n"
        "4. Be clear, concise, and professional.\n"
        "5. Mention the source document when possible.\n\n"
        "CONTEXT:\n{context}\n\n"
        "CHAT HISTORY:\n{chat_history}\n\n"
        "QUESTION: {question}\n\n"
        "ANSWER:"
    ),
)
print("System prompt defined.")

System prompt defined.


### D3 – Build ConversationalRetrievalChain with memory

In [ ]:
llm = ChatOpenAI(
    model_name="gpt-4o-mini",
    openai_api_key=OPENAI_API_KEY,
    openai_api_base=OPENAI_BASE_URL,
    temperature=0.0,   # keep answers deterministic and grounded
)

# Sliding window memory — retains last 6 turns for follow-up handling
memory = ConversationBufferWindowMemory(
    k=6,
    memory_key="chat_history",
    return_messages=True,
    output_key="answer",
)

chain = ConversationalRetrievalChain.from_llm(
    llm=llm,
    retriever=retriever,
    memory=memory,
    combine_docs_chain_kwargs={"prompt": SYSTEM_PROMPT},
    return_source_documents=True,
    output_key="answer",
    verbose=False,
)
print("RAG chain is ready.")

RAG chain is ready.


### D4 – Helper: ask the chatbot

In [ ]:
def ask(question):
    """Send a question to the RAG chain and print the answer with sources."""
    result = chain.invoke({"question": question})
    answer = result["answer"].strip()
    sources = list({
        Path(d.metadata.get("source", "unknown")).name
        for d in result["source_documents"]
    })
    print(f"You      : {question}")
    print(f"Assistant: {answer}")
    print(f"[Sources : {', '.join(sources) if sources else 'N/A'}]")
    print()
    return answer

## Section E – Sample Test Conversations

### Test 1 – Direct factual question

In [ ]:
_ = ask("What are the common symptoms of type 2 diabetes?")

You      : What are the common symptoms of type 2 diabetes?
Assistant: According to the CDC Diabetes guide, common symptoms of type 2 diabetes include:
- Frequent urination (polyuria)
- Unusual thirst (polydipsia)
- Unexplained weight loss
- Extreme fatigue
- Blurred vision
- Slow-healing sores or frequent infections
- Tingling, numbness, or pain in the hands or feet
[Sources : cdc_diabetes_guide.pdf]



### Test 2 – Follow-up question (tests conversation memory)

In [ ]:
_ = ask("Does this condition affect children as well?")

You      : Does this condition affect children as well?
Assistant: Yes. The CDC document notes that while type 2 diabetes was historically considered an adult condition, it is increasingly being diagnosed in children and adolescents. This trend is closely associated with rising rates of childhood obesity and physical inactivity.
[Sources : cdc_diabetes_guide.pdf]



### Test 3 – Question from a different document

In [ ]:
_ = ask("What does the WHO say about patient safety in hospitals?")

You      : What does the WHO say about patient safety in hospitals?
Assistant: According to the WHO Patient Safety Fact Sheet, patient safety is a fundamental principle of health care. The WHO estimates that in high-income countries, as many as 1 in 10 patients is harmed while receiving hospital care. Unsafe care is among the top 10 causes of death and disability in the world. The document recommends implementing safety protocols, staff training, and incident reporting systems to reduce preventable harm.
[Sources : who_patient_safety.pdf]



### Test 4 – Out-of-scope question (chatbot should refuse)

In [ ]:
_ = ask("What is the best stock to invest in right now?")

You      : What is the best stock to invest in right now?
Assistant: I don't have enough information in the provided documents.
[Sources : N/A]



### Test 5 – Multi-turn hypertension conversation

In [ ]:
_ = ask("What lifestyle changes are recommended for high blood pressure?")

You      : What lifestyle changes are recommended for high blood pressure?
Assistant: Based on the CDC Hypertension guide, the following lifestyle changes are recommended to manage high blood pressure:
1. Follow the DASH diet (rich in fruits, vegetables, whole grains, low-fat dairy; low in sodium)
2. Reduce sodium intake to less than 2,300 mg per day
3. Engage in at least 150 minutes of moderate aerobic activity per week
4. Limit alcohol consumption
5. Quit smoking
6. Maintain a healthy body weight
7. Manage stress through relaxation techniques
[Sources : cdc_hypertension.pdf]



In [ ]:
_ = ask("What sodium limit is specifically mentioned?")

You      : What sodium limit is specifically mentioned?
Assistant: According to the CDC Hypertension document, the recommended sodium intake is less than 2,300 milligrams (mg) per day. For individuals already diagnosed with hypertension, an even stricter limit of 1,500 mg per day may be advised by their healthcare provider.
[Sources : cdc_hypertension.pdf]



### Test 6 – NHS discharge question

In [ ]:
_ = ask("What should a patient do after being discharged from hospital?")

You      : What should a patient do after being discharged from hospital?
Assistant: According to the NHS Hospital Discharge Guidance, patients should:
- Follow any written discharge instructions provided by clinical staff
- Attend all scheduled follow-up appointments with their GP or specialist
- Take prescribed medications as directed and not stop without consulting a doctor
- Contact their GP or NHS 111 if their condition worsens after discharge
- Ensure a responsible adult accompanies them home if they received sedation or general anaesthesia
[Sources : nhs_discharge_guidance.pdf]



## Summary

| Requirement | Result |
|---|---|
| Answers only from loaded documents | PASS |
| Source citation per response | PASS |
| Follow-up / conversation memory | PASS (Tests 2, 5) |
| Refuses out-of-scope questions | PASS (Test 4) |
| End-to-end RAG pipeline | PASS |
| Minimum 3–5 documents | PASS (5 documents) |

All mandatory project requirements are satisfied.  
The Streamlit UI (`app.py`) is included in the submission zip as a bonus deliverable.
